# Exercise 17.2: LayerNorm & RMSNorm for Transformers

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vuhung16au/Maths-for-IT-Notebooks/blob/main/Chapter-17/02-layer-norm-and-rmsnorm.ipynb)

**Reference:** Chapter 17 — Feature Scaling, Normalisation and Dimensionality Reduction (`chapters/feature-scaling-and-pca.tex`)  
**Estimated Time:** 15–20 min  

### Learning Objectives
- Implement `layer_norm(x, gamma, beta, eps)` across the feature dimension (`axis=-1`).
- Implement `rms_norm(x, gamma, eps)` used in modern LLMs (LLaMA, Gemma).
- Compare mean-centring vs. root-mean-square scaling on token activation vectors.


In [ ]:
import numpy as np


### Mathematical Background

Inside Transformers and LLMs (Section 17.2), **LayerNorm** normalises each $d$-dimensional token vector $\mathbf{x} \in \mathbb{R}^d$ across its own features:

$$\operatorname{LayerNorm}(\mathbf{x}) = \gamma \odot \frac{\mathbf{x} - \mu}{\sqrt{\sigma^2 + \varepsilon}} + \beta$$

**RMSNorm** simplifies this by removing mean-centring and dividing only by the root-mean-square magnitude:

$$\operatorname{RMS}(\mathbf{x}) = \sqrt{\frac{1}{d}\sum_{i=1}^d x_i^2 + \varepsilon}, \qquad \operatorname{RMSNorm}(\mathbf{x}) = \gamma \odot \frac{\mathbf{x}}{\operatorname{RMS}(\mathbf{x})}$$


In [ ]:
def layer_norm(
    x: np.ndarray,
    gamma: np.ndarray | None = None,
    beta: np.ndarray | None = None,
    eps: float = 1e-5,
) -> np.ndarray:
    """Apply Layer Normalisation along the last axis.

    Formula:
        y = gamma * (x - mean) / sqrt(var + eps) + beta

    Args:
        x: Input array of shape (..., d).
        gamma: Optional scale parameter of shape (d,). Defaults to ones.
        beta: Optional shift parameter of shape (d,). Defaults to zeros.
        eps: Small constant for numerical stability.

    Returns:
        Normalised array of the same shape as x.
    """
    # TODO: Compute mean and variance along axis=-1 with keepdims=True, normalise, then apply gamma and beta
    raise NotImplementedError("Complete this TODO")


def rms_norm(
    x: np.ndarray,
    gamma: np.ndarray | None = None,
    eps: float = 1e-5,
) -> np.ndarray:
    """Apply Root Mean Square Normalisation (RMSNorm) along the last axis.

    Formula:
        rms = sqrt(mean(x^2) + eps)
        y = gamma * (x / rms)

    Args:
        x: Input array of shape (..., d).
        gamma: Optional scale parameter of shape (d,). Defaults to ones.
        eps: Small constant for numerical stability.

    Returns:
        RMS-normalised array of the same shape as x.
    """
    # TODO: Compute RMS along axis=-1 with keepdims=True, divide x by rms, and scale by gamma
    raise NotImplementedError("Complete this TODO")


<details>
<summary>Hint</summary>

- Use `axis=-1, keepdims=True` in `np.mean` and `np.var` so broadcasting works for both 1D `(d,)` and 2D `(seq_len, d)` inputs.
- Only multiply by `gamma` (and add `beta` in `layer_norm`) when they are not `None`.

</details>


In [ ]:
# 1. Single token vector
x = np.array([1.0, 2.0, 3.0, 4.0])
ln_x = layer_norm(x)
assert np.isclose(np.mean(ln_x), 0.0, atol=1e-7)
assert np.isclose(np.var(ln_x), 1.0, atol=1e-4)

rn_x = rms_norm(x)
assert np.isclose(np.sqrt(np.mean(rn_x ** 2)), 1.0, atol=1e-5)
# RMSNorm does not subtract the mean, so mean != 0
assert np.mean(rn_x) > 0.5

# 2. Batch of 3 tokens, d=4
rng = np.random.default_rng(42)
X_batch = rng.normal(loc=5.0, scale=3.0, size=(3, 4))
ln_batch = layer_norm(X_batch)
assert ln_batch.shape == (3, 4)
assert np.allclose(ln_batch.mean(axis=-1), 0.0, atol=1e-7)

# 3. Scale invariance of RMSNorm (with small eps)
assert np.allclose(rms_norm(10.0 * X_batch, eps=1e-12), rms_norm(X_batch, eps=1e-12), atol=1e-6)

print('All Chapter 17.2 assertions passed!')


### Think Deeper

1. Why does RMSNorm save computation compared to LayerNorm in large language models?
2. Why is LayerNorm shift-invariant ($\operatorname{LayerNorm}(\mathbf{x} + c) = \operatorname{LayerNorm}(\mathbf{x})$) whereas RMSNorm is only scale-invariant?
